# Notebook 2: Confirming the Bullwhip Effect

**Goal:** Reproduce and visualize the classic bullwhip effect — order variability
amplifies as you move upstream in a supply chain.

We will:
1. Show order amplification visually across echelons
2. Measure bullwhip ratios (variance amplification)
3. Demonstrate how lead time drives the bullwhip
4. Compare cumulative amplification across chain configurations
5. Validate with Monte Carlo simulation (1000 paths)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

SEED = 966  # Saudi country code

## 1. The Bullwhip Effect — Visual Proof

The bullwhip effect occurs when small fluctuations in customer demand get
**amplified** as orders propagate upstream through the supply chain.

Let's generate demand and simulate a 4-echelon chain to see this in action.

In [ ]:
from deepbullwhip import SemiconductorDemandGenerator, SerialSupplyChain

gen = SemiconductorDemandGenerator()
demand = gen.generate(T=156, seed=SEED)
fm = np.full_like(demand, demand.mean())
fs = np.full_like(demand, demand.std())

chain = SerialSupplyChain()
result = chain.simulate(demand, fm, fs)

### 1.1 Order Quantities Per Echelon

Watch how the order signal gets progressively wilder as you move upstream
(top = customer demand, bottom = raw material supplier):

In [ ]:
from deepbullwhip.diagnostics.plots import plot_order_quantities

fig = plot_order_quantities(demand, result)
plt.show()

### 1.2 Order Streams Overlay

Overlaying all order streams on a single plot makes the amplification stark:

In [ ]:
from deepbullwhip.diagnostics.plots import plot_order_streams

fig = plot_order_streams(demand, result)
plt.show()

## 2. Measuring the Bullwhip

The **bullwhip ratio** at echelon $k$ is:

$$BW_k = \frac{\text{Var}(\text{orders}_k)}{\text{Var}(\text{input}_k)}$$

where input is customer demand for E1 and downstream orders for E2+.
A ratio > 1 means orders are more variable than demand — i.e., the bullwhip.

In [ ]:
print(f"{'Echelon':<14} {'Var(orders)':>14} {'Var(input)':>14} {'BW Ratio':>10}")
print("-" * 56)

var_demand = np.var(demand)
for k, er in enumerate(result.echelon_results):
    var_orders = np.var(er.orders)
    if k == 0:
        var_input = var_demand
        input_label = "demand"
    else:
        var_input = np.var(result.echelon_results[k-1].orders)
        input_label = f"E{k} orders"
    print(f"E{k+1}: {er.name:<10} {var_orders:>14.2f} {var_input:>14.2f} {er.bullwhip_ratio:>10.3f}")

print(f"\nCumulative BW (E4 vs demand): {result.cumulative_bullwhip:.2f}")
print(f"  → Supplier orders are {result.cumulative_bullwhip:.0f}x more variable than demand")

### 2.1 Cumulative Amplification (Log Scale)

The cumulative bullwhip = $\prod_{k=1}^{K} BW_k$ shows exponential growth:

In [ ]:
from deepbullwhip.diagnostics.plots import plot_bullwhip_amplification

labels = [er.name for er in result.echelon_results]
fig = plot_bullwhip_amplification({"Default": result}, echelon_labels=labels)
plt.show()

## 3. What Drives the Bullwhip? Lead Time.

Under the Order-Up-To policy, the bullwhip ratio depends on lead time $L$.
Longer lead time → more safety stock → larger orders → more amplification.

Let's test this by creating chains with different lead time profiles.

In [ ]:
from deepbullwhip import EchelonConfig, default_semiconductor_config

# 3 scenarios: short, medium, long lead times
scenarios = {
    "Short (L=1,2,3,2)": [1, 2, 3, 2],
    "Default (L=2,4,12,8)": [2, 4, 12, 8],
    "Long (L=4,8,24,16)": [4, 8, 24, 16],
}

base = default_semiconductor_config()
results_lt = {}

for name, lead_times in scenarios.items():
    cfgs = [
        EchelonConfig(base[i].name, lead_times[i], base[i].holding_cost,
                      base[i].backorder_cost, base[i].depreciation_rate)
        for i in range(4)
    ]
    ch = SerialSupplyChain.from_config(cfgs)
    results_lt[name] = ch.simulate(demand, fm, fs)

fig = plot_bullwhip_amplification(results_lt, echelon_labels=labels)
plt.show()

In [ ]:
# Tabulate
import pandas as pd

rows = []
for name, res in results_lt.items():
    bw_vals = [er.bullwhip_ratio for er in res.echelon_results]
    rows.append({
        "Scenario": name,
        "BW E1": f"{bw_vals[0]:.2f}",
        "BW E2": f"{bw_vals[1]:.2f}",
        "BW E3": f"{bw_vals[2]:.2f}",
        "BW E4": f"{bw_vals[3]:.2f}",
        "Cumulative": f"{res.cumulative_bullwhip:.1f}",
    })

pd.DataFrame(rows).set_index("Scenario")

## 4. Bullwhip Across Service Levels

Higher service level → larger safety stock → larger order-up-to level S →
more order variability. Let's confirm:

In [ ]:
from deepbullwhip import default_semiconductor_config

results_sl = {}
for sl in [0.80, 0.90, 0.95, 0.99]:
    cfgs = [
        EchelonConfig(c.name, c.lead_time, c.holding_cost, c.backorder_cost,
                      c.depreciation_rate, service_level=sl)
        for c in default_semiconductor_config()
    ]
    ch = SerialSupplyChain.from_config(cfgs)
    results_sl[f"SL={sl:.0%}"] = ch.simulate(demand, fm, fs)

fig = plot_bullwhip_amplification(results_sl, echelon_labels=labels)
plt.show()

## 5. Monte Carlo Validation (1000 Paths)

A single demand path might be an outlier. Let's validate the bullwhip effect
is robust by running 1000 simulations using the vectorized engine.

In [ ]:
from deepbullwhip import VectorizedSupplyChain
import time

N = 1000
T = 156

t0 = time.perf_counter()
D_batch = gen.generate_batch(T=T, n_paths=N, seed=966)
fm_batch = np.full((N, T), D_batch.mean())
fs_batch = np.full((N, T), D_batch.std())

vchain = VectorizedSupplyChain()
mc_result = vchain.simulate(D_batch, fm_batch, fs_batch)
elapsed = time.perf_counter() - t0

print(f"Simulated {N} paths in {elapsed:.3f}s")
print(f"\nMean bullwhip ratios across {N} paths:")
for k in range(4):
    bw = mc_result.bullwhip_ratios[:, k]
    print(f"  E{k+1}: {bw.mean():.2f} (std={bw.std():.2f}, "
          f"min={bw.min():.2f}, max={bw.max():.2f})")
print(f"\nMean cumulative BW: {mc_result.cumulative_bullwhip.mean():.1f}")

In [ ]:
# Distribution of bullwhip ratios across paths
from deepbullwhip.diagnostics.plots import _apply_style, COLORS, DOUBLE_COL, GOLDEN, _echelon_color

_apply_style()
fig, axes = plt.subplots(1, 4, figsize=(DOUBLE_COL, DOUBLE_COL / GOLDEN / 2.2), sharey=True)

names = [c.name for c in default_semiconductor_config()]
for k, ax in enumerate(axes):
    bw = mc_result.bullwhip_ratios[:, k]
    ax.hist(bw, bins=30, color=_echelon_color(k), alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.axvline(bw.mean(), color='black', linestyle='--', linewidth=0.6)
    ax.set_xlabel('BW Ratio')
    ax.set_title(f'E{k+1}: {names[k]}', fontsize=8)

axes[0].set_ylabel('Count')
fig.suptitle(f'Bullwhip Ratio Distribution (N={N} paths)', fontsize=9, y=1.02)
plt.show()

## 6. Theoretical Lower Bound

Under the OUT policy with forecast sensitivity $\lambda_f$ and AR(1) demand
with parameter $\phi$, Theorem 1 gives:

$$BW_1 \geq 1 + \frac{2L\lambda_f \phi}{1 + \phi^2} + L^2 \lambda_f^2$$

This confirms the bullwhip is *structural* — it's a mathematical consequence
of the ordering policy, not a simulation artifact.

In [ ]:
from deepbullwhip.diagnostics.metrics import bullwhip_lower_bound

phi = 0.72  # AR(1) coefficient

print(f"{'L (lead time)':>14} {'lambda=0.1':>12} {'lambda=0.5':>12} {'lambda=1.0':>12}")
print("-" * 54)
for L in [1, 2, 4, 8, 12]:
    vals = [bullwhip_lower_bound(L, lam, phi) for lam in [0.1, 0.5, 1.0]]
    print(f"{L:>14d} {vals[0]:>12.2f} {vals[1]:>12.2f} {vals[2]:>12.2f}")

## Key Takeaways

1. **The bullwhip effect is real**: order variability amplifies exponentially
   across echelons — confirmed visually, numerically, and statistically
2. **Lead time is the primary driver**: doubling lead times roughly quadruples
   the cumulative bullwhip ratio
3. **Higher service level amplifies the bullwhip**: more safety stock means
   larger swings in order quantities
4. **It's not an artifact**: Monte Carlo over 1000 paths confirms the effect
   is robust, and the theoretical lower bound provides a mathematical guarantee